In [53]:
import numpy as np
import pandas as pd
import os
import plotly.express as px

In [118]:
def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.concat((weights_adv, [bias_adv]))

def calTheta(xP: np.array, theta0: np.array, alpha: float, methods: str):
    if methods == "ROAR":
        return calThetaAdv_linf(xP, theta0[:-1], theta0[-1], alpha)
    if methods == "Alg1":
        return calThetaAdv_linf(xP, theta0[:-1], theta0[-1], alpha)
    if methods == "L1PSD":
        return calThetaAdv_l1(xP, theta0, alpha)

def getStats(xP: np.ndarray, x0: np.ndarray, theta: np.ndarray, lamb):
    if xP.size != theta.size:
        x0 = np.hstack((x0, 1))
        xP = np.hstack((xP, 1))

    return np.log(1 + np.exp(-(xP @ theta))) + \
        (lamb * (np.linalg.norm(x0 - xP, ord=1)))

In [152]:
def readPickle(final_path: str):
    df = pd.read_pickle(final_path)
    df["theta_r"] = df.apply(lambda row : calTheta(row['x_r'], row['theta_0'], row['alpha'], row['algorithm']), axis=1)
    df['J'] = df.apply(lambda row: getStats(row['x_r'], row['x_0'], row['theta_r'], row['lambda']), axis=1)

    return df

def readPickleAll(dir_path: str, dataset_name: str):
    all_files = []

    for file in os.listdir(dir_path):
        if dataset_name in file.split(sep='_'):
            all_files.append(os.path.join(dir_path, file))
    df = pd.concat([pd.read_pickle(f_n) for f_n in all_files])

    df["theta_r"] = df.apply(lambda row : calTheta(row['x_r'], row['theta_0'], row['alpha'], row['algorithm']), axis=1)
    df['J'] = df.apply(lambda row: getStats(row['x_r'], row['x_0'], row['theta_r'], row['lambda']), axis=1)
    df = df.groupby("algorithm", group_keys=False).apply(lambda x : x.reset_index(drop=True))

    return df

In [162]:
dir_path = "../results/recourse"
dataset_name = "sba"

output = readPickleAll(dir_path, dataset_name)

C:\Users\pmyat\AppData\Local\Temp\ipykernel_8660\2381527330.py:18: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [163]:
output_mean = output.groupby(['algorithm', 'seed'], as_index=False).mean()
output_mean

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0,theta_r,J
0,Alg1,0,0.5,0.1,19.0,"[0.2467282051282052, -0.33360512820512844, 0.4...","[0.2467282051282052, -0.33360512820512844, 0.4...","[0.3521, -0.2441000000000001, 0.04600000000000...","[0.2367153846153847, -0.07743333333333331, -0....",0.559415
1,Alg1,1,0.5,0.1,17.5,"[-0.12308611111111108, -0.08241666666666665, 0...","[-0.12308611111111108, -0.08241666666666665, 0...","[0.3199, -0.15589999999999996, -0.035, 0.1301,...","[0.5143444444444447, -0.10034444444444443, -0....",0.515511
2,Alg1,3,0.5,0.1,17.5,"[0.19422777777777778, -0.33530277777777795, 0....","[0.19422777777777778, -0.33530277777777795, 0....","[0.11209999999999996, -0.17929999999999985, 0....","[0.05654444444444437, 0.01514444444444443, -0....",0.519709
3,L1PSD,0,0.5,0.1,19.0,"[0.2467282051282052, -0.33360512820512844, 0.4...","[0.2465461538461539, -0.3339179487179487, 0.45...","[0.3521, -0.2441000000000001, 0.04600000000000...","[0.33927948717948725, -0.25692051282051287, 0....",0.318604
4,L1PSD,1,0.5,0.1,17.5,"[-0.12308611111111108, -0.08241666666666665, 0...","[-0.12261944444444445, -0.08150833333333336, 0...","[0.3199, -0.15589999999999996, -0.035, 0.1301,...","[0.3199000000000001, -0.15589999999999996, -0....",0.284407
5,L1PSD,3,0.5,0.1,17.5,"[0.19422777777777778, -0.33530277777777795, 0....","[0.19413055555555556, -0.3372583333333332, 0.5...","[0.11209999999999996, -0.17929999999999985, 0....","[0.11209999999999999, -0.20707777777777758, 0....",0.301873
6,ROAR,0,0.5,0.1,19.0,"[0.2467282051282052, -0.33360512820512844, 0.4...","[0.5180332477276142, -0.606697522676908, 0.449...","[0.3521001277825771, -0.24409998380220854, 0.0...","[0.18543324103722206, -0.051792291494516224, -...",2.723961
7,ROAR,1,0.5,0.1,17.5,"[-0.12308611111111108, -0.08241666666666665, 0...","[0.3508333100212945, -0.41460559103224015, 0.2...","[0.3198998769124349, -0.1559000015258789, -0.0...","[0.5143446392483182, -0.10034444597032335, -0....",1.104318
8,ROAR,3,0.5,0.1,17.5,"[0.19422777777777778, -0.33530277777777795, 0....","[0.33916393915812176, -0.5478694703843858, 0.4...","[0.11209993892245823, -0.1792999505996704, 0.0...","[0.02876667512787713, 0.09847780068715413, -0....",1.141142


In [110]:
px.scatter(output, y="J", color="algorithm", facet_col="algorithm")

In [165]:
px.scatter(output_mean, x = "seed", y = "J", color="algorithm", 
           labels = {"seed" : "Fold", "J" : "Total Cost"},
           title=f"{dataset_name} Dataset Recourse Mean" )

In [170]:
output.groupby("algorithm").mean().pct_change()['J']

algorithm
Alg1          NaN
L1PSD   -0.432486
ROAR     4.578944
Name: J, dtype: float64

In [122]:
dir_path = "../results/recourse"
file_name = "lr_synthetic_Alg1_1.pkl"
final_path = os.path.join(dir_path, file_name)

df1 = readPickle(final_path)

In [123]:
dir_path = "../results/recourse"
file_name = "lr_synthetic_L1PSD_1.pkl"
final_path = os.path.join(dir_path, file_name)

# df2 = pd.read_pickle(final_path)
# # df2 = df2.rename(columns={"x_r": "theta_0", "theta_0": "x_r"})
# df2['theta_r'] = df2.apply(lambda row : calThetaAdv_l1(np.hstack((row['x_r'], 1)), row['theta_0'], row['alpha']), axis=1)
# df2['J'] = df2.apply(lambda row: getStats(row['x_r'], row['x_0'], row['theta_r'], row['lambda']), axis=1)
# # df2.loc[:,['algorithm', 'seed', 'alpha', 'lambda', 'i', 'x_0', 'x_r', 'theta_0']]

# # df2.to_pickle(final_path)

# df2

df2 = readPickle(final_path)

In [ ]:
dir_path = "../results/recourse"
file_name = "lr_synthetic_ROAR_0.pkl"
final_path = os.path.join(dir_path, file_name)

df3 = pd.read_pickle(final_path)

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0
0,ROAR,0,0.5,0.1,0,"[-2.3862, -1.7774]","[0.3794, 0.8222]","[1.9661, 1.9713, 0.0506]"
1,ROAR,0,0.5,0.1,1,"[-2.7105, -1.7402]","[0.2405, 0.9448]","[1.9661, 1.9713, 0.0506]"
2,ROAR,0,0.5,0.1,2,"[-2.3817, -1.9247]","[0.4369, 0.7677]","[1.9661, 1.9713, 0.0506]"
3,ROAR,0,0.5,0.1,3,"[-2.337, -1.0178]","[0.0955, 1.0694]","[1.9661, 1.9713, 0.0506]"
4,ROAR,0,0.5,0.1,4,"[-2.402, -2.7282]","[0.7175, 0.4867]","[1.9661, 1.9713, 0.0506]"
...,...,...,...,...,...,...,...,...
91,ROAR,0,0.5,0.1,91,"[-1.7741, -2.2538]","[0.7767, 0.4302]","[1.9661, 1.9713, 0.0506]"
92,ROAR,0,0.5,0.1,92,"[-2.2471, -3.4363]","[1.0089, 0.1598]","[1.9661, 1.9713, 0.0506]"
93,ROAR,0,0.5,0.1,93,"[-1.2901, -2.4369]","[1.0113, 0.1709]","[1.9661, 1.9713, 0.0506]"
94,ROAR,0,0.5,0.1,94,"[-2.911, -1.7206]","[0.1547, 1.0144]","[1.9661, 1.9713, 0.0506]"
